In [35]:
import pyreadr  # Lê arquivos .rds

from transformers import pipeline
from sentence_transformers import SentenceTransformer

# Pre-processamento dos dados

In [33]:
# Lê os arquivos de dados (rds)
medicamentos_rds = pyreadr.read_r("../../../data/pncp/medicamentos.rds")
catmat_rds = pyreadr.read_r("../../../data/catmat/catmat-para-python.rds")  # utilize o catmat-para-python.rds (sem colunas aninhadas)

# Extrai o dataframe
medicamentos_df = medicamentos_rds[None]
catmat_df = catmat_rds[None]

# Verifica número de registros
print(f'Número de medicamentos (PNCP): {len(medicamentos_df)}')
print(f'Número de medicamentos (CATMAT): {len(catmat_df)}')

Número de medicamentos (PNCP): 144967
Número de medicamentos (CATMAT): 5771


In [ ]:
# Visualiza o dataframe de medicamentos
medicamentos_df.head(5)


,endpoint,numeroItem,descricao,materialOuServico,materialOuServicoNome,valorUnitarioEstimado,valorTotal,quantidade,unidadeMedida,orcamentoSigiloso,...,temResultado,imagem,aplicabilidadeMargemPreferenciaNormal,aplicabilidadeMargemPreferenciaAdicional,percentualMargemPreferenciaNormal,percentualMargemPreferenciaAdicional,ncmNbsCodigo,ncmNbsDescricao,clean_descricao,codigo_pdm
0,https://pncp.gov.br/api/pncp/v1/orgaos/0040255...,8.0,Cloreto de potássio,M,Material,283.24,283.24,1.0,Quilograma,False,...,True,0.0,NaN,NaN,NaN,NaN,NaN,NaN,cloreto de potassio,5116
1,https://pncp.gov.br/api/pncp/v1/orgaos/0040255...,9.0,Acetato De Cálcio,M,Material,845.37,845.37,1.0,Quilograma,False,...,False,0.0,NaN,NaN,NaN,NaN,NaN,NaN,acetato de calcio,1733
2,https://pncp.gov.br/api/pncp/v1/orgaos/0039454...,1.0,Álcool Etílico,M,Material,20.12,1207.20,60.0,Frasco 1000 ML,False,...,True,0.0,NaN,NaN,NaN,NaN,NaN,NaN,alcool etilico,2259
3,https://pncp.gov.br/api/pncp/v1/orgaos/0039454...,2.0,Álcool Etílico,M,Material,10.30,370.80,36.0,Frasco 1000 ML,False,...,True,0.0,NaN,NaN,NaN,NaN,NaN,NaN,alcool etilico,2259
4,https://pncp.gov.br/api/pncp/v1/orgaos/0039454...,1.0,Contraste Radiológico,M,Material,53.32,56999.08,1069.0,Frasco 50 ML,False,...,False,0.0,NaN,NaN,NaN,NaN,NaN,NaN,contraste radiologico,5804


In [27]:
# Visualiza o dataframe do CATMAT
catmat_df.head(3)

,codigo_grupo,nome_grupo,codigo_classe,nome_classe,codigo_pdm,nome_pdm,codigo_br,nome_item,desc_item,item_suspenso,item_ativo,desc_item_ativo,item_sustentavel,desc_item_sustentavel
0,65,"EQUIPAMENTOS E ARTIGOS PARA USO MÉDICO, DENTÁR...",6505,DROGAS E MEDICAMENTOS,14597,Petrolato,233632,"Petrolato, Aspecto Físico:Líquido, Tipo:Laxati...","233632 - Petrolato, Aspecto Físico:Líquido, Ti...",False,True,Ativo,False,NÃO SUSTENTAVEL
1,65,"EQUIPAMENTOS E ARTIGOS PARA USO MÉDICO, DENTÁR...",6505,DROGAS E MEDICAMENTOS,8325,Imunoglobulina Humana,260160,"Imunoglobulina Humana, Tipo:Hiper Imuni Para H...","260160 - Imunoglobulina Humana, Tipo:Hiper Imu...",False,True,Ativo,False,NÃO SUSTENTAVEL
2,65,"EQUIPAMENTOS E ARTIGOS PARA USO MÉDICO, DENTÁR...",6505,DROGAS E MEDICAMENTOS,3924,Fenoterol Bromidrato,266532,"Fenoterol Bromidrato, Dosagem:0,2mg / Dose, Ap...","266532 - Fenoterol Bromidrato, Dosagem:0,2mg /...",False,True,Ativo,False,NÃO SUSTENTAVEL


### Medicamentos PNCP

In [37]:
# Seleciona colunas úteis 
medicamentos_df = medicamentos_df[['endpoint', 'numeroItem', 'codigo_pdm', 'descricao', 'unidadeMedida']].copy()

# Deixa a descricao e unidadeMedida em minusculo
medicamentos_df['descricao'] = medicamentos_df['descricao'].astype(str).str.lower()
medicamentos_df['unidadeMedida'] = medicamentos_df['unidadeMedida'].astype(str).str.lower()

# Une a descrição e a unidade de medida, separando-as por ponto e vírgula (;)
medicamentos_df['descricaoCompleta'] = medicamentos_df['descricao'] + '; ' + medicamentos_df['unidadeMedida']

medicamentos_df.head(5)

,endpoint,numeroItem,codigo_pdm,descricao,unidadeMedida,descricaoCompleta
0,https://pncp.gov.br/api/pncp/v1/orgaos/0040255...,8.0,5116,cloreto de potássio,quilograma,cloreto de potássio; quilograma
1,https://pncp.gov.br/api/pncp/v1/orgaos/0040255...,9.0,1733,acetato de cálcio,quilograma,acetato de cálcio; quilograma
2,https://pncp.gov.br/api/pncp/v1/orgaos/0039454...,1.0,2259,álcool etílico,frasco 1000 ml,álcool etílico; frasco 1000 ml
3,https://pncp.gov.br/api/pncp/v1/orgaos/0039454...,2.0,2259,álcool etílico,frasco 1000 ml,álcool etílico; frasco 1000 ml
4,https://pncp.gov.br/api/pncp/v1/orgaos/0039454...,1.0,5804,contraste radiológico,frasco 50 ml,contraste radiológico; frasco 50 ml


### Catálogo de materiais (CATMAT)

In [29]:
# Seleciona colunas úteis 
catmat_df = catmat_df[['codigo_br', 'desc_item']].copy()

# Deixa a descricao em minusculo
catmat_df['desc_item'] = catmat_df['desc_item'].astype(str).str.lower()

catmat_df.head(5)

,codigo_br,desc_item
0,233632,"233632 - petrolato, aspecto físico:líquido, ti..."
1,260160,"260160 - imunoglobulina humana, tipo:hiper imu..."
2,266532,"266532 - fenoterol bromidrato, dosagem:0,2mg /..."
3,266665,"266665 - arteméter, dosagem:80 mg/ml, apresent..."
4,266699,"266699 - budesonida, apresentação:aerossol buc..."


# De texto para Embeddings

In [39]:
# Carrega o modelo
# Mais informações em: https://huggingface.co/Snowflake/snowflake-arctic-embed-l-v2.0
model_name = 'Snowflake/snowflake-arctic-embed-l-v2.0'
model = SentenceTransformer(model_name)

In [ ]:
# Define as consultas e os documentos
consultas = medicamentos_df['descricaoCompleta']
documentos = catmat_df['desc_item']

# Computa os embeddings: use `prompt_name="query"` para codificar (encode) as consultas
query_embeddings = model.encode(consultas, prompt_name="query") 
document_embeddings = model.encode(documentos)

# Calcula a similaridade de coseno entre as consultas e os documentos
scores = model.similarity(consultas, documentos)

to be continued...

# Testando 1,2,3

Isso é um teste simples com um classificador de frases (zero-shot classification)

In [42]:
classifier = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli")

In [4]:
sequence_to_classify = "VACINA ANTIRRÁBICA PARA CÃES. Auxilia na prevenção das infecções do vírus da raiva. Forma farmacêutica: suspensão injetável. Com seringa."
candidate_labels = [
    "Vacina, Composição:Raiva (Cultivado Em Células Vero), Tipo:Inativada, Forma Farmaceutica:Pó Liófilo P/ Injetável + Diluente",
    "Vacina, Composição:Meningocócica B, Tipo:Recombinante, Forma Farmaceutica:Suspensão Injetável",
    "Vacina, Composição:Dengue 1, 2, 3, 4, Tipo:Atenuada, Forma Farmaceutica:Injetável",
    "Dipirona Sódica, Dosagem:500 MG"]
output = classifier(sequence_to_classify, candidate_labels, multi_label=False)
print(output)

{'sequence': 'VACINA ANTIRRÁBICA PARA CÃES. Auxilia na prevenção das infecções do vírus da raiva. Forma farmacêutica: suspensão injetável. Com seringa.', 'labels': ['Vacina, Composição:Dengue 1, 2, 3, 4, Tipo:Atenuada, Forma Farmaceutica:Injetável', 'Vacina, Composição:Meningocócica B, Tipo:Recombinante, Forma Farmaceutica:Suspensão Injetável', 'Vacina, Composição:Raiva (Cultivado Em Células Vero), Tipo:Inativada, Forma Farmaceutica:Pó Liófilo P/ Injetável + Diluente', 'Dipirona Sódica, Dosagem:500 MG'], 'scores': [0.6841481328010559, 0.26933127641677856, 0.045952796936035156, 0.000567856477573514]}
